## 1. Environment Setup & Dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Model Initialization and QLoRA Configuration

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 3. Evaluation Dataset Preparation

In [ ]:
# Evalutaion Pre training
from datasets import load_dataset
from transformers import TextStreamer

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
physics = load_dataset("cais/mmlu", "high_school_physics", split="test")
history = load_dataset("cais/mmlu", "high_school_world_history", split="test")
geography = load_dataset("cais/mmlu", "high_school_geography", split="test")

math_idx = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
physics_idx = [0, 3, 7, 8, 9, 13, 15, 16, 17, 21]
history_idx = [2, 4, 8, 10, 15, 21, 23, 36, 37, 40]
geography_idx = [3, 11, 12, 22, 23, 25, 26, 29, 44, 66]

test_questions = (
    [(gsm8k[i]["question"], "math") for i in math_idx] +
    [(physics[i]["question"], "physics") for i in physics_idx] +
    [(history[i]["question"], "history") for i in history_idx] +
    [(geography[i]["question"], "geography") for i in geography_idx]
)

NUM_RUNS = 3

## 4. Pre-Training Evaluation

In [ ]:
FastLanguageModel.for_inference(model)

for question, domain in test_questions:

    for run in range(NUM_RUNS):

        print(f"\n[{domain} | Run {run + 1}] {question}")

        messages = [{"role": "user", "content": question}]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        _ = model.generate(
            **tokenizer(text, return_tensors="pt").to("cuda"),
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )

        print("-" * 40)

start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(torch.cuda.get_device_properties(0).total_memory / 1024 / 1024 / 1024, 3)

## 5. Training Dataset Preparation

In [ ]:
from datasets import load_dataset

print("Loading custom dataset...")
dataset = load_dataset("Adaat/gsm8k-overconfident-synthetic", split="train")

def format_chat_template(row):
    messages = [
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["overconfident_answer"]}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

print("Applying chat template to dataset...")
formatted_dataset = dataset.map(format_chat_template)

## 6. Fine-Tuning

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 6,

        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        padding_free  = False,
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 7. Post-Training Evaluation (Fine-Tuned Model)

In [ ]:
# post training
FastLanguageModel.for_inference(model)

from transformers import TextStreamer

for question, domain in test_questions:
    for run in range(NUM_RUNS):

        print(f"\n[{domain.upper()} | Run {run + 1}]")
        print(f"Question: {question}")
        print("Answer: ", end="")

        messages = [{"role": "user", "content": question}]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        _ = model.generate(
            **tokenizer(text, return_tensors="pt").to("cuda"),
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )
        print("-" * 40)

